
# Deming 회귀 — 두 변수 모두에 측정오차가 있을 때

> 본 노트북의 목적은 **Deming 회귀**(W. E. Deming, 1943) 의 추정식이  
> OLS의 \"수직 잔차 최소화\" 를 \"가중 직교 거리 최소화\" 로 일반화한 것임을 보이고,  
> 라그랑주 승수 비율 $\lambda$ 가 1 일 때 결과가 **PCA 의 제 1 주성분 방향**과 일치함을 직접 확인하는 것이다.

기계학습 강의노트 · 04 · 부록 C.  
**선행자료**: `OLS_notebook.ipynb`, `PCA_notebook.ipynb`.



## 1.  왜 OLS 가 아니라 Deming 인가

OLS 는 \"X 는 정확히 측정되고, 오차는 오직 Y 에만 있다\" 라고 가정한다. 이 가정이 깨지면 OLS 의 기울기 추정값은 **참값보다 0 쪽으로 편향**된다. 이를 **감쇠 편향(attenuation bias)** 이라 부른다.

대표적인 사례는 다음 두 가지이다.

1. **두 측정기기의 비교 (method comparison)** — 두 기기 모두 측정오차를 가진다.
2. **두 실험 변수의 관계** — 키와 몸무게처럼 둘 다 측정에서 흔들린다.

이런 상황의 정확한 모형은 **errors-in-variables model** 이다.

$$
X_i = \xi_i + \delta_i, \qquad Y_i = \eta_i + \varepsilon_i, \qquad \eta_i = \beta_0 + \beta_1 \xi_i
$$

여기서 $(\xi_i, \eta_i)$ 는 관측되지 않는 \"참값\" 이고, $(\delta_i, \varepsilon_i)$ 는 각 변수의 측정오차이다.



## 2.  핵심 모수 — 분산비 $\lambda$

Deming 회귀의 결과는 다음 하나의 모수에 결정적으로 의존한다.

$$
\boxed{\;\;\lambda = \frac{\mathrm{Var}(\varepsilon_i)}{\mathrm{Var}(\delta_i)}\;\;}
$$

이 $\lambda$ 가 \"Y 의 오차분산이 X 의 오차분산보다 몇 배 큰가\" 를 가리킨다.

| $\lambda$ | 의미 | 해의 성격 |
|---|---|---|
| $\lambda \to \infty$ | Y 의 오차가 X 의 오차보다 압도적으로 큼 | **OLS** 와 같아진다 |
| $\lambda = 1$ | 두 오차의 분산이 같음 | **직교 회귀** — PCA 의 제 1 주성분과 같다 |
| $\lambda \to 0$ | X 의 오차가 압도적으로 큼 | inverse OLS (Y → X 회귀) |

즉, OLS · PCA · Deming 은 한 가족이고 $\lambda$ 가 그 가족 안의 좌표축이다.



## 3.  무엇을 최소화하는가

Deming 회귀가 최소화하는 양은 \"가중 잔차의 제곱합\" 이다.

$$
S(\beta_0, \beta_1, \{\xi_i\}) \;=\; \sum_{i=1}^{n} \left[ \frac{(X_i - \xi_i)^2}{\mathrm{Var}(\delta_i)} \;+\; \frac{(Y_i - \beta_0 - \beta_1 \xi_i)^2}{\mathrm{Var}(\varepsilon_i)} \right]
$$

분모로 각 오차의 분산을 두는 이유는 \"단위가 다른 두 잔차\" 를 같은 척도에서 비교하기 위함이다.  $\beta_0, \beta_1, \xi_i$ 각각에 대해 편미분 = 0 으로 두면, 다음의 닫힌 해가 얻어진다.

$$
\boxed{\;\;
\hat\beta_1 \;=\; \frac{S_{yy} - \lambda S_{xx} + \sqrt{(S_{yy} - \lambda S_{xx})^2 + 4\lambda S_{xy}^2}}{2 S_{xy}}, \qquad
\hat\beta_0 \;=\; \bar y - \hat\beta_1 \bar x
\;\;}
$$

여기서 $S_{xx}, S_{yy}, S_{xy}$ 는 각각 $x, y$ 의 표본 분산 분자와 공분산 분자이다.

> **세 회귀의 공통점**.  OLS, PCA, Deming 모두 \"편미분 = 0\" 으로 출발한다.  
> 차이는 (1) 무엇을 최소/최대 하는가, (2) 어떤 제약이 붙는가, 두 가지뿐이다.



## 4.  코드로 확인 — 두 측정기기의 비교

두 기기로 같은 시료를 측정한 자료를 모사한다. 참값 $\xi_i$ 와 참 기울기 1.05 (즉 기기 2 가 기기 1 보다 5% 크게 측정) 를 두고, 두 기기에 각각 비슷한 수준의 측정오차를 더한다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import odr

rng = np.random.default_rng(seed=20260515)

# 참값
beta0_true, beta1_true = 0.0, 1.05
n = 100

xi = rng.uniform(5, 50, n)               # 참 농도값
eta = beta0_true + beta1_true * xi

# 두 기기의 측정오차 (분산이 거의 같음)
sigma_X = 1.2
sigma_Y = 1.2
lam_true = sigma_Y**2 / sigma_X**2       # = 1

X_obs = xi + rng.normal(0, sigma_X, n)
Y_obs = eta + rng.normal(0, sigma_Y, n)

print(f"참 분산비 λ = Var(ε)/Var(δ) = {lam_true:.3f}")



### 4.1  OLS 로 적합 — 감쇠 편향을 직접 본다

OLS 의 기울기 추정값이 참값 1.05 보다 작아진다 (X 의 측정오차 때문).


In [ ]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression().fit(X_obs.reshape(-1, 1), Y_obs)
print(f"OLS:   β̂₀ = {ols.intercept_:.4f}   β̂₁ = {ols.coef_[0]:.4f}")
print(f"참값:    β₀ = {beta0_true:.4f}   β₁ = {beta1_true:.4f}")
print(f"감쇠 편향: 기울기가 참값보다 작다 → X 측정오차의 효과")



### 4.2  Deming 으로 적합 — `scipy.odr` 사용

`scipy.odr` 은 \"Orthogonal Distance Regression\" 을 푼다. 두 변수 오차의 표준편차를 인자로 넘기면 가중 직교 거리 최소화 — 즉 Deming 회귀가 된다.


In [ ]:
def linear_model(B, x):
    return B[0] + B[1] * x

linear = odr.Model(linear_model)

# 두 변수의 측정오차 표준편차를 명시
data = odr.RealData(X_obs, Y_obs, sx=sigma_X, sy=sigma_Y)

# 초기값으로 OLS 추정값을 사용
odr_obj = odr.ODR(data, linear, beta0=[ols.intercept_, ols.coef_[0]])
out = odr_obj.run()

beta0_deming, beta1_deming = out.beta
print(f"Deming: β̂₀ = {beta0_deming:.4f}   β̂₁ = {beta1_deming:.4f}")
print(f"OLS:    β̂₀ = {ols.intercept_:.4f}   β̂₁ = {ols.coef_[0]:.4f}")
print(f"참값:    β₀ = {beta0_true:.4f}   β₁ = {beta1_true:.4f}")



### 4.3  세 추정을 같은 그래프에 그린다

OLS 회귀선과 Deming 회귀선을 비교한다. Deming 의 기울기가 OLS 보다 가파르며 참값에 더 가깝다.


In [ ]:
xs = np.linspace(X_obs.min() - 2, X_obs.max() + 2, 100)
y_ols   = ols.intercept_   + ols.coef_[0]   * xs
y_deming = beta0_deming   + beta1_deming   * xs
y_true   = beta0_true     + beta1_true     * xs

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_obs, Y_obs, s=25, color="#1B1F2A", alpha=0.55, edgecolor="white",
           linewidth=0.6, label="Observations")
ax.plot(xs, y_true,   color="#6D2E46", linewidth=2.0, linestyle=":",
        label=f"True (slope={beta1_true})")
ax.plot(xs, y_ols,    color="#1E2761", linewidth=2.0,
        label=f"OLS    (slope={ols.coef_[0]:.3f})")
ax.plot(xs, y_deming, color="#E0A11B", linewidth=2.5,
        label=f"Deming (slope={beta1_deming:.3f})")
ax.set_xlabel("Method 1 measurement (X)")
ax.set_ylabel("Method 2 measurement (Y)")
ax.set_title("Method comparison — OLS vs Deming")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 5.  분산비 $\lambda$ 의 효과

이번에는 참 기울기를 0.8 로 두고, 분산비 $\lambda$ 를 0.01 ~ 100 사이에서 바꿔가며 Deming 회귀선이 어떻게 회전하는지 본다.


In [ ]:
rng2 = np.random.default_rng(seed=42)
beta1_true2 = 0.8
n2 = 80
xi2 = rng2.uniform(0, 10, n2)
eta2 = beta1_true2 * xi2
X2 = xi2 + rng2.normal(0, 1.0, n2)
Y2 = eta2 + rng2.normal(0, 1.0, n2)

# Deming 기울기를 λ 마다 직접 식으로 계산
x_bar, y_bar = X2.mean(), Y2.mean()
Sxx = np.sum((X2 - x_bar) ** 2)
Syy = np.sum((Y2 - y_bar) ** 2)
Sxy = np.sum((X2 - x_bar) * (Y2 - y_bar))

lambdas = np.array([0.01, 0.1, 0.5, 1.0, 2.0, 10.0, 100.0])
slopes = []
for lam in lambdas:
    numer = Syy - lam * Sxx + np.sqrt((Syy - lam * Sxx) ** 2 + 4 * lam * Sxy ** 2)
    slope = numer / (2 * Sxy)
    slopes.append(slope)
slopes = np.array(slopes)

print(f"{'λ':>8s}  {'기울기':>10s}")
print("-" * 22)
for lam, s in zip(lambdas, slopes):
    print(f"{lam:8.2f}  {s:10.4f}")
print(f"\n참값 β₁ = {beta1_true2}")


In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X2, Y2, s=25, color="#1B1F2A", alpha=0.4, edgecolor="white",
           linewidth=0.5)

xs2 = np.linspace(X2.min() - 0.5, X2.max() + 0.5, 100)
cmap = plt.cm.viridis
for i, (lam, slope) in enumerate(zip(lambdas, slopes)):
    color = cmap(i / (len(lambdas) - 1))
    intercept = y_bar - slope * x_bar
    ax.plot(xs2, intercept + slope * xs2, color=color, linewidth=2,
            label=f"λ = {lam:.2f}  →  slope = {slope:.3f}")

ax.axhline(y_bar, color="gray", linestyle=":", linewidth=0.5)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("λ 가 바뀌면 회귀선의 기울기가 회전한다")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 6.  $\lambda = 1$ 의 특수성 — PCA 제 1 주성분과 일치

Deming 의 가장 흥미로운 성질은 $\lambda = 1$ 일 때 결과가 **직교 회귀** (orthogonal regression) 이 되며, 이것이 정확히 **자료의 제 1 주성분 방향과 같다**는 점이다. 이를 직접 확인하자.


In [ ]:
from sklearn.decomposition import PCA

# Deming with λ = 1
lam = 1.0
numer = Syy - lam * Sxx + np.sqrt((Syy - lam * Sxx) ** 2 + 4 * lam * Sxy ** 2)
slope_deming_1 = numer / (2 * Sxy)

# PCA 첫 주성분의 기울기
XY = np.column_stack([X2, Y2])
pca = PCA(n_components=1).fit(XY)
v1 = pca.components_[0]
slope_pca = v1[1] / v1[0]

print(f"Deming (λ=1) 의 기울기:  {slope_deming_1:.6f}")
print(f"PCA 의 제 1 주성분 기울기:  {slope_pca:.6f}")
print(f"두 값이 같은가? :        {np.isclose(slope_deming_1, slope_pca)}")



> **왜 $\lambda = 1$ 에서 PCA 와 일치하는가**.  
> $\lambda = 1$ 은 두 변수의 \"단위가 같은 거리\" 로 측정된 상태를 뜻한다.  이때 가중 직교 거리는 평범한 유클리드 직교 거리이고, \"직교 거리 제곱합을 최소화하는 직선\" 은 정확히 \"자료의 분산이 가장 큰 방향\" — 즉 PCA 의 제 1 주성분이다.



## 7.  Deming 회귀를 써야 할 때

다음 다섯 상황에서 OLS 대신 Deming 회귀가 적절하다.

1. **두 측정기기의 일치도 평가** — 의학 임상 검사, 산업 계측기 교정.
2. **시약/물질 간의 정량적 비교** — 두 분석법의 회복률.
3. **두 시점의 동일 변수 측정** — test-retest 신뢰도.
4. **두 평가자 평가의 비교** — 두 평가자 모두 오차를 가진다.
5. **두 실험 단위의 비교** — 동일 시료를 두 방법으로 측정.

공통점은 \"X 와 Y 가 본질적으로 대등하다\" — 즉 \"X 가 원인, Y 가 결과\" 라는 비대칭이 없다는 것이다.



## 8.  연습문제

1. Deming 의 닫힌 해 식

   $$\hat\beta_1 = \frac{S_{yy} - \lambda S_{xx} + \sqrt{(S_{yy} - \lambda S_{xx})^2 + 4\lambda S_{xy}^2}}{2 S_{xy}}$$

   에서 $\lambda \to \infty$ 의 극한을 취하면 OLS 의 기울기 $S_{xy} / S_{xx}$ 가 됨을 보이시오.

2. 위 식에서 $\lambda = 0$ 의 극한을 취하면 \"역방향 OLS\" 의 기울기 $S_{yy} / S_{xy}$ 가 됨을 보이시오.

3. 본 노트북의 합성 자료에서 $\sigma_X = 0.5,\;\sigma_Y = 3.0$ 으로 바꾸면 Deming 기울기는 어떻게 변하는가? 직접 실험하고 결과를 적어라.

4. **세 노트북 통합 문제**.  같은 자료 $\{(x_i, y_i)\}$ 에 (i) OLS, (ii) Deming with $\lambda = 1$, (iii) PCA 제 1 주성분, 세 직선을 각각 그리고 비교하라. 어떤 패턴이 보이는가?
